## Практика 1 (новости Вышки)

Давайте обкачаем немного новостей с сайта вышки.

1. Страницы имеют вид "https://www.hse.ru/news/page1.html", поэтому можно просто идти циклом.
2. Достанем дату публикации, заголовок, краткое описание (из страницы со списком новостей), текст полной статьи и метки (из самой страницы новости)
3. Сохраним в датафрейм.

In [2]:
!pip install fake_useragent

In [75]:
from bs4 import BeautifulSoup
import re
import requests
from fake_useragent import UserAgent
import time
from tqdm import tqdm
from urllib.parse import urljoin
import pandas as pd

ua = UserAgent()
session = requests.session()

### **Шаг 1. Найти страницы**

Посмотрим, как устроены новости и скачаем одну страницу:

In [60]:
page_number = 1
url = f'https://www.hse.ru/news/page{page_number}.html'
req = session.get(url)
#req = session.get(url, headers={'User-Agent': ua.random})
page = req.text

In [61]:
soup = BeautifulSoup(page, 'lxml')

Найдем отдельные посты:

In [62]:
news = soup.find_all('div', {'class': 'post'})

In [31]:
len(news)

10

In [32]:
print(news[0].prettify())

<div class="post post_first">
 <div class="post__extra">
  <div class="post-meta">
   <div class="post-meta__filter-ico rubric-white rubric_3">
    <svg height="70" viewbox="0 0 70 70" width="70" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink">
     <rect clip-path="url(#hse-mask)" fill="#0941AF" height="70" width="70" x="0" y="0">
     </rect>
     <image class="svg" clip-path="url(#hse-mask)" height="70" src="/f/src/global/i/mask-rubric-cut.png" width="70" xlink:href="/f/src/global/i/mask-rubric-cut.png"/>
    </svg>
    <span class="filter-ico filter-ico_mask">
     <img height="70" src="/f/src/global/i/mask-pic.png" width="70"/>
    </span>
   </div>
   <div class="post-meta__date">
    <div class="post-meta__day">
     13
    </div>
    <div class="post-meta__month">
     ноя
    </div>
    <div class="post-meta__year">
     2026
    </div>
    <div class="post-meta__visitors">
    </div>
   </div>
  </div>
 </div>
 <div class="post__content">
  <div 

Найдем заголовок-ссылку и запомним текст заголовка:

In [33]:
title_obj = news[0].find('a')
title_obj.text

'«Сейчас музеи хантят сотрудников из корпоративной среды»'

Достанем свойства этой ссылки (куда ведет, `class`):

In [34]:
title_obj.attrs

{'href': 'https://www.hse.ru/success/filippova',
 'class': ['link', 'link_dark2', 'no-visited']}

Достанем саму ссылку:

In [35]:
title_obj.attrs['href']

'https://www.hse.ru/success/filippova'

Достанем текст новости:

In [36]:
news[0].find('div', {'class': 'post__text'}).text

'Выпускница НИУ ВШЭ Дарья Филиппова занимается культурными инициативами Банка ВТБ. Четыре года назад под ее руководством в Москве открылся Центр «Зотов», а в следующем году откроется после реставрации Дом Радио в Санкт-Петербурге. В интервью «Конструктору успеха» Дарья рассказала о связи социологии с искусством, своем опыте перехода из музейной в корпоративную среду и о том, что у нее общего с выставкой «Работать и жить».'

Достанем день, месяц, год публикации:

In [37]:
pub_day = news[0].find('div', {'class': 'post-meta__day'}).text
pub_day

'13'

In [38]:
pub_month = news[0].find('div', {'class': 'post-meta__month'}).text
pub_month

'ноя'

In [39]:
pub_year = news[0].find('div', {'class': 'post-meta__year'}).text
pub_year

'2026'

### **Шаг 2. Научиться парсить страничку самой новости**

Скачаем и распарсим полученную ссылку:

In [49]:
link = 'https://www.hse.ru/news/edu/1217901538.html'

In [50]:
req = session.get(link)
page = req.text

soup = BeautifulSoup(page, 'html.parser')

In [51]:
full_text = soup.find('div', {'class': 'post__content'}).text
full_text[:200]

'Вышка открыла представительство на Ближнем Востоке и подписала соглашение с вузом №1 в ОАЭЕвгения Кочева, Анна Тышецкая, Никита Анисимов (справа налево)© Высшая школа экономикиВ Абу-Даби открылось офи'

Найдем теги, которые присвоены статье:

In [52]:
meta = soup.find('div', {'class': 'articleMeta'})

tags = meta.find_all('a', {'class': 'tag'})
tags = [t.text for t in tags]
tags

['международное сотрудничество', 'Вышка глобальная']

In [53]:
rubrics = meta.find_all('a', {'class': 'rubric'})
rubrics = [r.text for r in rubrics]
rubrics

['Образование']

In [59]:
people = meta.find_all('span', {'class': 'b link'})
people = [p.text for p in people]
people

['Анисимов Никита Юрьевич', 'Тышецкая Анна Юрьевна']

### **Шаг 3. Оформляем нормально в функции**

Парсим информацию из страницы со списком новостей (блок одной новости):

In [76]:
def parse_news_page_block(one_block):
    block = {}
    a = one_block.find('a')
    block['title'] = a.text
    block['href'] = a.attrs['href']
    block['short_text'] = one_block.find('div', {'class': 'post__text'}).text
    block['pub_day'] = int(one_block.find('div', {'class': 'post-meta__day'}).text)
    block['pub_month'] = one_block.find('div', {'class': 'post-meta__month'}).text
    block['pub_year'] = int(one_block.find('div', {'class': 'post-meta__year'}).text)
    return block

Парсим отдельную страницу новости:

In [77]:
def parse_one_article(block):
    url_one = urljoin('https://www.hse.ru', block['href'])
    req = session.get(url_one)
    page = req.text
    soup = BeautifulSoup(page, 'lxml')
    block['full_text'] = soup.find('div', {'class': 'post__content'}).text
    meta = soup.find('div', {'class': 'articleMeta'})
    tags = meta.find_all('a', {'class': 'tag'})
    block['tags'] = [t.text for t in tags]
    return block

Регулярное выражение для того, чтобы достать ID новости и не повторяться:

In [81]:
regex_hse_id = re.compile('/([0-9]*?).html')

Обработать N-ую страницу новостей:

In [78]:
def get_nth_page(page_number):
    # скачиваем
    url = f'https://www.hse.ru/news/page{page_number}.html'
    req = session.get(url)
    time.sleep(3)
    page = req.text
    soup = BeautifulSoup(page, 'lxml')

    # находим новости
    news = soup.find_all('div', {'class': 'post'})

    # идем по новостям и обрабатываем их
    blocks = []
    for n in news:
        try:
            blocks.append(parse_news_page_block(n))
        except Exception as e:
            print('ОШИБКА', e)

    # идем по отдельным статьям и достаем информацию
    result = []
    for b in blocks:
        if b['href'].startswith('/'):
            idx = regex_hse_id.findall(b['href'])[0]
            try:
                res = parse_one_article(b)
                time.sleep(3)
                res['hse_id'] = idx
                result.append(res)
            except Exception as e:
                print(e)

    # возвращаем найденную информацию
    return result

### **Шаг 4. Сохраняем в датафрейм**


Напишем функцию, куда передаем количество страниц и она выполняет все нужные действия:

In [79]:
def run_all(n_pages):
    blocks = []
    for i in tqdm(range(n_pages)):
        blocks.extend(get_nth_page(i+1))

    return blocks

Запускаем на 3 первых страниц:

In [82]:
blocks = run_all(3)

df = pd.DataFrame(blocks)

100%|██████████| 3/3 [02:03<00:00, 41.07s/it]


In [83]:
df

,title,href,short_text,pub_day,pub_month,pub_year,full_text,tags,hse_id
0,НИУ ВШЭ и ИФТТ РАН заключили соглашение о созд...,/news/science/1217988522.html,Церемония подписания соглашения о создании Лаб...,13,ноя,2026,НИУ ВШЭ и ИФТТ РАН заключили соглашение о созд...,[],1217988522
1,ИИ и практика: ВШБ ВШЭ представила в Индонезии...,/news/1217966293.html,Члены Ассоциации бизнес-школ Азиатско-Тихоокеа...,13,ноя,2026,ИИ и практика: ВШБ ВШЭ представила в Индонезии...,[репортаж о событии],1217966293
2,Вышка открыла представительство на Ближнем Вос...,/news/edu/1217901538.html,В Абу-Даби открылось официальное представитель...,13,ноя,2026,Вышка открыла представительство на Ближнем Вос...,"[международное сотрудничество, Вышка глобальная]",1217901538
3,Город как общее дело: чему поколения урбанисто...,/news/expertise/1217700503.html,Российская урбанистика складывалась на пересеч...,12,ноя,2026,Город как общее дело: чему поколения урбанисто...,[],1217700503
4,"«Формулирование теорий — последний навык, в ко...",/news/science/1217696101.html,"Луи Вервурт, доцент МИЭМ ВШЭ, специалист в обл...",12,ноя,2026,"«Формулирование теорий — последний навык, в ко...",[исследования и аналитика],1217696101
5,"«Космос рядом»: космические технологии, инжене...",/news/life/1217666622.html,19–20 ноября 2026 года в Москве на площадке НИ...,12,ноя,2026,"«Космос рядом»: космические технологии, инжене...",[приглашение к участию],1217666622
6,Ученики и учителя российских школ изучили возм...,/news/edu/1217406961.html,В конце октября учебный центр НИУ ВШЭ «Воронов...,11,ноя,2026,Ученики и учителя российских школ изучили возм...,"[репортаж о событии, довузовская подготовка, д...",1217406961
7,Студенты ФСН ВШЭ — в числе победителей конкурс...,/news/edu/1217208071.html,Студенты 3-го курса программы «Вычислительные ...,11,ноя,2026,Студенты ФСН ВШЭ — в числе победителей конкурс...,"[достижения, студенты]",1217208071
8,Обновленная образовательная модель обеспечит к...,/news/edu/1217317646.html,Высшая школа экономики обновляет и развивает с...,11,ноя,2026,Обновленная образовательная модель обеспечит к...,[новое в ВШЭ],1217317646
9,Географы НИУ ВШЭ разработали новый цифровой ин...,/news/expertise/1217215058.html,Цифровой инструмент для экспресс-оценки природ...,11,ноя,2026,Географы НИУ ВШЭ разработали новый цифровой ин...,[новое в ВШЭ],1217215058


## Практика-2 (letterboxd)

Сделайте краулер фильмов аналогичный тому, что выше.

У вас есть список на [Letterboxd](https://letterboxd.com/) c перечислением всех фильмов из книжки *1001 Movies You Must See Before You Die*: https://letterboxd.com/peterstanley/list/1001-movies-you-must-see-before-you-die/detail/.

1. Все страницы выглядят как `https://letterboxd.com/peterstanley/list/1001-movies-you-must-see-before-you-die/detail/page/<номер>/`, поэтому снова можно идти циклом.
2. Достанем номер фильма, его название, год выпуска со страницы списка.
3. Достанем режиссера, слоган (если есть) и краткое описание с самой страницы фильма.
4. Сохраним в датафрейм.